# JurisEC — Notebook 4: Significance Tests Between Systems

**Goal.** Check whether the performance differences between the four systems evaluated on JurisEC (BM25, E5-large, EmbeddingGemma-300M, and BGE-M3) are statistically significant or could be due to chance.

**Question.** For each pair of systems: if both performed equally well, how often would a difference as large as the observed one appear by chance?

**Method.**
- Per-query metrics: nDCG@10 and Acc@1, computed on the same 5,536 queries and 5,508 candidate documents.
- Two-sided paired randomization test (10,000 permutations) for the six pairs of systems.
- Holm correction for multiple comparisons, applied within each metric.
- 95% bootstrap confidence intervals (2,000 resamples) for each system and for each pairwise difference.

**How to read the results.** An adjusted p-value below 0.05 means the difference is statistically significant. Otherwise, the two systems cannot be told apart with these data.

**Requirements.** A GPU and the `HF_TOKEN` secret in Colab (EmbeddingGemma is a gated model). The notebook is self-contained: it downloads the dataset and computes the embeddings, or reuses them if they are already in `cache_emb/`.

**Outputs** (in `results/`): `significance_pairwise.csv` (the six comparisons) and `significance_confidence_intervals.csv` (the intervals for each system).

**Reproducibility note.** The tests use a fixed random seed. Embeddings are computed in half precision, so scores may vary in the third decimal place depending on the hardware.


In [1]:
%%capture
!pip install -q -U datasets rank_bm25 transformers sentence-transformers


In [2]:
import os, re, json, unicodedata
import numpy as np, pandas as pd, torch
from datasets import load_dataset

SEED   = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE  = "/content/cache_emb"
OUT    = "/content/results"
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)

BLOQUE = 512
K      = 10

# Hugging Face token: required for EmbeddingGemma (a gated model)
# and for the dataset if the repository is private.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("No Hugging Face token found in the Colab secrets (HF_TOKEN):", type(e).__name__)
print("Device:", DEVICE)


Device: cuda


## 1. Data and relevance judgments

Loads the 5,536 queries with their official responses. Identical responses are collapsed into a single candidate document, which leaves 5,508 documents. Nearly identical queries (TF-IDF cosine similarity of 0.90 or higher) receive reciprocal relevance judgments: the response to one also counts as relevant for the other.


In [3]:
ds = load_dataset("wilfredomartel/JurisEC")
ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
df = ds.to_pandas()

def limpia_texto(x):
    return x.strip() if isinstance(x, str) else ""

for col in ["consulta", "resolucion_consulta", "materia", "fuente"]:
    if col in df.columns: df[col] = df[col].map(limpia_texto)
df = df[(df["consulta"] != "") & (df["resolucion_consulta"] != "")].reset_index(drop=True)

texto_a_docid, corpus_ids, corpus_texts = {}, [], []
for t in df["resolucion_consulta"]:
    if t not in texto_a_docid:
        texto_a_docid[t] = f"d{len(corpus_texts)}"
        corpus_ids.append(texto_a_docid[t]); corpus_texts.append(t)
df["doc_id"] = df["resolucion_consulta"].map(texto_a_docid)
df["qid"]    = [f"q{i}" for i in range(len(df))]

qids     = df["qid"].tolist()
n_q, n_d = len(qids), len(corpus_ids)
pos_doc  = {d: i for i, d in enumerate(corpus_ids)}
doc_de_oro = df["doc_id"].map(pos_doc).to_numpy()
print(f"Queries: {n_q:,} | Unique documents: {n_d:,} | Collapsed duplicates: {len(df)-n_d}")
if (n_q, n_d) != (5536, 5508):
    print("WARNING: the paper reports 5,536 queries and 5,508 documents; this dataset gives different counts.")

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/evaluation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.2MB            

data/evaluation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating evaluation split:   0%|          | 0/5536 [00:00<?, ? examples/s]

Queries: 5,536 | Unique documents: 5,508 | Collapsed duplicates: 28


In [4]:
def normaliza(t):
    '''Lowercases the text and removes accents.'''
    t = unicodedata.normalize("NFD", t.lower())
    return "".join(c for c in t if unicodedata.category(c) != "Mn")

# Nearly identical query pairs (TF-IDF cosine similarity >= 0.90).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vect = TfidfVectorizer(preprocessor=normaliza, token_pattern=r"[a-z0-9]{3,}")
M = vect.fit_transform(df["consulta"].tolist())
pares_gemelas = []
for ini in range(0, M.shape[0], 512):
    sim = np.asarray(cosine_similarity(M[ini:ini+512], M))
    for fila in range(sim.shape[0]):
        i = ini + fila; sim[fila, i] = -1.0
        j = int(sim[fila].argmax())
        if sim[fila, j] >= 0.90 and i < j: pares_gemelas.append((i, j))

qrels = {q: {d: 1} for q, d in zip(qids, df["doc_id"])}
for i, j in pares_gemelas:
    qrels[qids[i]][df["doc_id"].iloc[j]] = 1
    qrels[qids[j]][df["doc_id"].iloc[i]] = 1
json.dump(qrels, open(f"{OUT}/qrels.json", "w"))

relevantes = [np.array(sorted(pos_doc[d] for d in qrels[q]), dtype=np.int32) for q in qids]
n_rel      = np.array([len(r) for r in relevantes])
print(f"Nearly identical query pairs: {len(pares_gemelas)}")
print(f"Queries with more than one relevant document: {(n_rel > 1).sum()} ({(n_rel > 1).mean():.1%})")

Nearly identical query pairs: 29
Queries with more than one relevant document: 58 (1.0%)


## 2. Embeddings for the dense models

Encodes the queries and documents with E5-large, EmbeddingGemma-300M, and BGE-M3, using each model's maximum input length and prefixes. Embeddings already saved in `cache_emb/` are reused.


In [5]:
from sentence_transformers import SentenceTransformer

queries_texts = df["consulta"].tolist()
MODELOS = {     # settings for each model
    "BGE-M3":         dict(repo="BAAI/bge-m3", archivos=("BGE-M3_Q.npy", "BGE-M3_D.npy"),
                           tokens=8192, lote=8,  pref_q="", pref_d=""),
    "E5-large":       dict(repo="intfloat/multilingual-e5-large", archivos=("E5_Q.npy", "E5_D.npy"),
                           tokens=512,  lote=32, pref_q="query: ", pref_d="passage: "),
    "EmbeddingGemma": dict(repo="google/embeddinggemma-300m", archivos=("GEMMA_Q.npy", "GEMMA_D.npy"),
                           tokens=2048, lote=16, pref_q="task: search result | query: ",
                           pref_d="title: none | text: "),
}

def cache_valida(cfg):
    fq, fd = (f"{CACHE}/{a}" for a in cfg["archivos"])
    if not (os.path.exists(fq) and os.path.exists(fd)): return False
    return (np.load(fq, mmap_mode="r").shape[0] == n_q and np.load(fd, mmap_mode="r").shape[0] == n_d)

for nombre, cfg in MODELOS.items():
    if cache_valida(cfg):
        print(f"{nombre}: embeddings already saved; not recomputed."); continue
    assert DEVICE == "cuda", ("Embeddings are missing and no GPU is available. Enable a GPU in "
                              "Runtime > Change runtime type.")
    if nombre == "EmbeddingGemma":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    else:
        dtype = torch.float16
    modelo = SentenceTransformer(cfg["repo"], device=DEVICE, model_kwargs={"torch_dtype": dtype})
    modelo.max_seq_length = cfg["tokens"]
    if nombre == "EmbeddingGemma":      # prompts must match the official model card
        assert modelo.prompts.get("query") == cfg["pref_q"], modelo.prompts
        assert modelo.prompts.get("document") == cfg["pref_d"], modelo.prompts
    def codifica(textos):
        return modelo.encode(textos, batch_size=cfg["lote"], normalize_embeddings=True,
                             show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    np.save(f"{CACHE}/{cfg['archivos'][0]}", codifica([cfg["pref_q"] + q for q in queries_texts]))
    np.save(f"{CACHE}/{cfg['archivos'][1]}", codifica([cfg["pref_d"] + d for d in corpus_texts]))
    del modelo; torch.cuda.empty_cache()
    print(f"{nombre}: embeddings computed and saved to {CACHE}.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (48 outputs left out)

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

Batches:   0%|          | 0/346 [00:00<?, ?it/s]

Batches:   0%|          | 0/345 [00:00<?, ?it/s]

EmbeddingGemma: embeddings computed and saved to /content/cache_emb.


## 3. Per-query metrics

Ranks all 5,508 documents for each query and computes nDCG@10 and Acc@1 per query for the four systems. The table compares the mean nDCG@10 with the value in Table 6 of the paper and should show `ok` for all four systems.


In [6]:
STOPWORDS = set(("de la el en y a los del las un una que por con para es al se su o e lo como mas "
"pero sus le ya fue este si porque esta entre cuando muy sin sobre tambien me hasta hay donde quien "
"desde todo nos durante todos uno les ni contra otros ese eso ante ellos esto mi antes algunos unos "
"yo otro otras otra tanto esa estos mucho quienes nada muchos cual sea poco ella estar haber estas "
"estaba estamos algunas algo nosotros").split())


In [7]:
ARCHIVOS = {"E5-large":       ("E5_Q.npy",     "E5_D.npy"),
            "EmbeddingGemma": ("GEMMA_Q.npy",  "GEMMA_D.npy"),
            "BGE-M3":         ("BGE-M3_Q.npy", "BGE-M3_D.npy")}
SISTEMAS = ["BM25", "E5-large", "EmbeddingGemma", "BGE-M3"]

def fuente_densa(nombre):
    fq, fd = ARCHIVOS[nombre]
    Q = np.load(f"{CACHE}/{fq}").astype(np.float32)
    D = np.load(f"{CACHE}/{fd}").astype(np.float32)
    assert Q.shape[0] == n_q and D.shape[0] == n_d, (Q.shape, D.shape, n_q, n_d)
    for ini in range(0, n_q, BLOQUE):
        yield ini, Q[ini:ini+BLOQUE] @ D.T

def fuente_bm25():
    from rank_bm25 import BM25Okapi
    def tokenize(texto):
        return [w for w in re.findall(r"[a-z0-9]+", normaliza(texto))
                if len(w) > 2 and w not in STOPWORDS]
    indice = BM25Okapi([tokenize(d) for d in corpus_texts])
    consultas_tok = [tokenize(c) for c in df["consulta"]]
    for ini in range(0, n_q, BLOQUE):
        yield ini, np.vstack([indice.get_scores(t)
                              for t in consultas_tok[ini:ini+BLOQUE]]).astype(np.float32)

def fuente(nombre):
    return fuente_bm25() if nombre == "BM25" else fuente_densa(nombre)

DESCUENTO = 1.0 / np.log2(np.arange(2, K + 2))
IDCG      = np.array([DESCUENTO[:min(K, r)].sum() for r in range(0, n_d + 1)][:K + 1])

def recorrer(nombre):
    rango  = np.empty(n_q, np.int32)
    r1     = np.empty(n_q, np.float64)
    ndcg10 = np.empty(n_q, np.float64)
    for ini, S in fuente(nombre):
        for f in range(S.shape[0]):
            i, fila = ini + f, S[f]
            rel   = relevantes[i]
            s_oro = fila[rel].max()
            rango[i] = 1 + int((fila > s_oro).sum())
            top = np.argpartition(-fila, K)[:K]
            top = top[np.argsort(-fila[top], kind="stable")]
            ganancias = np.isin(top, rel).astype(np.float64)
            r1[i]     = ganancias[0] / len(rel)
            ndcg10[i] = (ganancias * DESCUENTO).sum() / IDCG[min(K, len(rel))]
    return dict(rango=rango, exito1=(rango == 1), r1=r1, ndcg10=ndcg10)

R = {}
TABLE_6 = {"BM25": 0.767, "E5-large": 0.753, "EmbeddingGemma": 0.782, "BGE-M3": 0.774}
print(f"{'System':<16}{'nDCG@10':>9}{'Table 6':>9}{'Acc@1':>8}{'R@1':>8}")
for s in SISTEMAS:
    R[s] = recorrer(s)
    ok = "ok" if abs(R[s]["ndcg10"].mean() - TABLE_6[s]) < 5e-3 else "CHECK!"
    print(f"{s:<16}{R[s]['ndcg10'].mean():>9.4f}{TABLE_6[s]:>9.3f}"
          f"{R[s]['exito1'].mean():>8.4f}{R[s]['r1'].mean():>8.4f}  {ok}")

System            nDCG@10  Table 6   Acc@1     R@1
BM25               0.7671    0.767  0.6581  0.6522  ok
E5-large           0.7527    0.753  0.6479  0.6439  ok
EmbeddingGemma     0.7810    0.782  0.6794  0.6749  ok
BGE-M3             0.7745    0.774  0.6676  0.6632  ok


## 4. Significance tests and confidence intervals

For each pair of systems, computes the mean difference, its 95% bootstrap confidence interval (2,000 resamples), and the p-value of a two-sided paired randomization test (10,000 permutations), adjusted with the Holm correction within each metric. The results are saved to two CSV files.


In [8]:
import itertools
rng_s = np.random.default_rng(13)
B_PERM, B_BOOT = 10_000, 2000
NOMBRE = {"ndcg10": "nDCG@10", "exito1": "Acc@1"}

def comparar(a, b):
    """Mean difference A-B, 95% paired bootstrap CI, and two-sided paired randomization p-value."""
    d = a.astype(float) - b.astype(float); n = len(d); obs = d.mean()
    boot = np.array([d[rng_s.integers(0, n, n)].mean() for _ in range(B_BOOT)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    perm = np.array([(d * rng_s.choice([-1.0, 1.0], n)).mean() for _ in range(B_PERM)])
    return obs, lo, hi, ((np.abs(perm) >= abs(obs)).sum() + 1) / (B_PERM + 1)

filas = []
for metrica in ["ndcg10", "exito1"]:
    for s1, s2 in itertools.combinations(SISTEMAS, 2):
        filas.append((NOMBRE[metrica], s1, s2, *comparar(R[s1][metrica], R[s2][metrica])))
res = pd.DataFrame(filas, columns=["metric", "A", "B", "A-B", "CI low", "CI high", "p"])

for m in res["metric"].unique():              # Holm correction within each metric
    orden = res[res["metric"] == m]["p"].sort_values().index
    acum = 0.0
    for r, i in enumerate(orden):
        acum = max(acum, min(1.0, res.loc[i, "p"] * (len(orden) - r)))
        res.loc[i, "p (Holm)"] = acum
res["significant (0.05)"] = np.where(res["p (Holm)"] < 0.05, "yes", "no")
print(res.round(4).to_string(index=False))
res.to_csv(f"{OUT}/significance_pairwise.csv", index=False)

print("\n95% bootstrap confidence interval for each system")
ic = []
for s in SISTEMAS:
    for metrica in ["ndcg10", "exito1"]:
        x = R[s][metrica].astype(float)
        boot = np.array([x[rng_s.integers(0, len(x), len(x))].mean() for _ in range(B_BOOT)])
        lo, hi = np.percentile(boot, [2.5, 97.5]); ic.append((s, NOMBRE[metrica], x.mean(), lo, hi))
        print(f"{s:<16}{NOMBRE[metrica]:<9}{x.mean():.4f}  [{lo:.4f}, {hi:.4f}]")
pd.DataFrame(ic, columns=["system", "metric", "mean", "CI low", "CI high"]).to_csv(f"{OUT}/significance_confidence_intervals.csv", index=False)
print(f"\nSaved to {OUT}/significance_pairwise.csv and {OUT}/significance_confidence_intervals.csv")


 metric              A              B     A-B  CI low  CI high      p  p (Holm) significant (0.05)
nDCG@10           BM25       E5-large  0.0144  0.0059   0.0232 0.0011    0.0033                yes
nDCG@10           BM25 EmbeddingGemma -0.0139 -0.0216  -0.0058 0.0007    0.0028                yes
nDCG@10           BM25         BGE-M3 -0.0074 -0.0152   0.0004 0.0665    0.0718                 no
nDCG@10       E5-large EmbeddingGemma -0.0283 -0.0351  -0.0218 0.0001    0.0006                yes
nDCG@10       E5-large         BGE-M3 -0.0218 -0.0281  -0.0152 0.0001    0.0006                yes
nDCG@10 EmbeddingGemma         BGE-M3  0.0065  0.0002   0.0125 0.0359    0.0718                 no
  Acc@1           BM25       E5-large  0.0101 -0.0023   0.0231 0.1257    0.2462                 no
  Acc@1           BM25 EmbeddingGemma -0.0213 -0.0334  -0.0092 0.0008    0.0032                yes
  Acc@1           BM25         BGE-M3 -0.0096 -0.0213   0.0022 0.1231    0.2462                 no
  Acc@1   